In [ ]:
import os
from spatialdata.models import Image2DModel
import numpy as np
import tifffile
import spatialdata as sd
import pandas as pd
from skimage.measure import regionprops_table
import spatialdata_plot
import spatialdata_io
import matplotlib.pyplot as plt
import seaborn as sns
import zarr
from copy import copy
import matplotlib.colors as mcolors
import dask.array as da
import skimage.measure
import xmltodict
from spatialdata import SpatialData
import spatialdata as sd


/data1/lowes/reyesj3/miniconda3/envs/spatialdata_opencv/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data1/lowes/reyesj3/miniconda3/envs/spatialdata_opencv/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
import importlib.util
import sys
def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))
imaging_utils =  lazy_import("imaging_utils",os.path.join(utils_dir, "imaging_utils.py"))
from general_utils import ismember, grep, grep_exclude
from imaging_utils import imnormalize, im2rgb, imblend, imoverlay


In [ ]:
SLIDE_ID = "SLIDE-0294"
rawdatapath = "../rawdata"
selected_subslide = "R000"

current_slide_path = os.path.join(rawdatapath, SLIDE_ID + "_Final")

possible_files = np.sort(os.listdir(current_slide_path))
possible_files = grep_exclude("0.0.2", possible_files)
possible_files = grep_exclude("0.1", possible_files)
possible_files = grep_exclude("Antibody1", possible_files)
possible_files = grep_exclude("2.0.4_" + selected_subslide + "_DAPI", possible_files)
possible_files = grep(selected_subslide, possible_files)

In [ ]:
possible_files

array(['SLIDE-0294_1.0.4_R000_Cy3_p16-EPR-IIry-555_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_1.0.4_R000_Cy5_uPAR-IIry-647_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_1.0.4_R000_DAPI__FINAL_F.ome.tif',
       'SLIDE-0294_1.0.4_R000_FITC_TNC-IIry-488_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_2.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_2.0.4_R000_Cy5_HMGA2-D1A7-647_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_2.0.4_R000_Cy7_PanCK-ae1-ae3-750_FINAL_AFR_F.ome.tif',
       'SLIDE-0294_2.0.4_R000_FITC_GFP--488_FINAL_AFR_F.ome.tif'],
      dtype='<U63')

In [ ]:
selected_channels = ["p16", "uPAR", "DAPI", "TNC", "Arg1", "HMGA2", "PanCK", "GFP"]
all_images = []
for my_channel in selected_channels:
    ome_tiff_file = os.path.join(current_slide_path, grep(my_channel, possible_files)[0])
    img_zarr = tifffile.imread(
        ome_tiff_file,
        aszarr=True,
        level=0,
    )
    img_da = da.from_zarr(img_zarr)
    all_images.append(img_da)


/data1/lowes/reyesj3/miniconda3/envs/sopa/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


In [ ]:
img_da = da.stack(all_images)
scale_factors = [1,2,4,8]
sdImage = sd.models.Image2DModel.parse(img_da, scale_factors=scale_factors, c_coords=selected_channels, dims=["c", "y", "x"])

In [ ]:
sdata = SpatialData({'celldive': sdImage})

In [ ]:
sdata_path = "../sdata"
path_output = os.path.join(sdata_path, SLIDE_ID)
sdata.write(path_output, overwrite = True)